# 05 — Logistic Regression From Scratch (Sections 13, 14)

Synthetic binary classification (2 features), manual logits/probabilities/loss/gradient descent, and evaluation metrics.

**Compliance (from-scratch boundary):** no sklearn classifiers/metrics, no `torch.optim.*`, no `torch.nn.Module`/`nn.Linear`/`nn.Parameter`, no `nn` losses/modules, no `torch.nn.functional` loss/objective helpers, no `torch.utils.data` abstractions. Uses raw tensors, explicit math (incl. numerically stable logits-based BCE), manual updates in `torch.no_grad()`.


In [1]:
import os
import numpy as np
import torch
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

torch.manual_seed(7)
np.random.seed(7)

os.makedirs('../reports', exist_ok=True)
print('reports dir ready')


reports dir ready


## Section 13 — Binary Classification From Scratch
### 1. Generate the features
### 2. Generate binary target labels
### 3. Visualize the two classes
### 4. Split the data into training and validation sets
### 5. Define trainable weights and bias
### 6. Calculate logits
### 7. Convert logits into probabilities for evaluation
### 8. Calculate binary classification loss using a numerically stable logits-based loss
### 9. Calculate gradients
### 10. Update parameters through gradient descent


In [2]:
# 1-2. Generate 2D features and binary labels
n_per_class = 150
mean0 = torch.tensor([-2.0, -1.0])
mean1 = torch.tensor([2.0, 1.5])
cov_scale = 0.8
x0 = mean0 + torch.randn(n_per_class, 2) * cov_scale
x1 = mean1 + torch.randn(n_per_class, 2) * cov_scale
X = torch.cat([x0, x1], dim=0)
y = torch.cat([torch.zeros(n_per_class, 1), torch.ones(n_per_class, 1)], dim=0)

perm = torch.randperm(X.shape[0])
X = X[perm]
y = y[perm]
print('X shape:', tuple(X.shape), 'y shape:', tuple(y.shape))

# 3. Visualize
plt.figure(figsize=(5,4))
plt.scatter(X[y.squeeze()==0,0].numpy(), X[y.squeeze()==0,1].numpy(), s=15, alpha=0.7, label='class 0')
plt.scatter(X[y.squeeze()==1,0].numpy(), X[y.squeeze()==1,1].numpy(), s=15, alpha=0.7, label='class 1')
plt.xlabel('x1')
plt.ylabel('x2')
plt.title('Synthetic binary classification data')
plt.legend()
plt.tight_layout()
plt.savefig('../reports/section13_classes.png', dpi=120)
plt.show()

# 4. Train/val split
n_train = int(0.8 * X.shape[0])
X_train, y_train = X[:n_train], y[:n_train]
X_val, y_val = X[n_train:], y[n_train:]
print('Train:', tuple(X_train.shape), 'Val:', tuple(X_val.shape))

# 5. Trainable weights and bias (raw tensors)
W = torch.randn(2, 1, requires_grad=True)
b = torch.zeros(1, requires_grad=True)

def logits(X_in, W_in, b_in):
  return X_in @ W_in + b_in

def sigmoid(z):
  return torch.sigmoid(z)

def stable_bce_with_logits(z, y_true):
  # Numerically stable: max(z,0) - z*y + log(1 + exp(-|z|))
  return (torch.clamp(z, min=0) - z * y_true + torch.log(1 + torch.exp(-torch.abs(z)))).mean()

lr = 0.1
epochs = 300
train_losses = []
val_losses = []
for epoch in range(epochs):
  z = logits(X_train, W, b)
  loss = stable_bce_with_logits(z, y_train)
  loss.backward()
  with torch.no_grad():
    W -= lr * W.grad
    b -= lr * b.grad
    W.grad.zero_()
    b.grad.zero_()
  train_losses.append(loss.item())
  with torch.no_grad():
    z_val = logits(X_val, W, b)
    val_loss = stable_bce_with_logits(z_val, y_val)
    val_losses.append(val_loss.item())

print('Final train loss:', train_losses[-1], 'Final val loss:', val_losses[-1])
plt.figure(figsize=(5,4))
plt.plot(train_losses, label='train')
plt.plot(val_losses, label='val')
plt.xlabel('Epoch')
plt.ylabel('BCE loss (logits)')
plt.title('Logistic regression training')
plt.legend()
plt.tight_layout()
plt.savefig('../reports/section13_training_loss.png', dpi=120)
plt.show()

with torch.no_grad():
  probs_val = sigmoid(logits(X_val, W, b))
print('Example probabilities (first 5):', probs_val[:5].squeeze().tolist())


X shape: (300, 2) y shape: (300, 1)


/tmp/ipykernel_69101/785645823.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Train: (240, 2) Val: (60, 2)
Final train loss: 0.015394655987620354 Final val loss: 0.013460657559335232


Example probabilities (first 5): [0.0028348902706056833, 0.9958628416061401, 0.9950113296508789, 0.9762444496154785, 1.4282874872151297e-05]


/tmp/ipykernel_69101/785645823.py:77: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Section 14 — Evaluation and Decision Boundary
### 1. Evaluate the logistic regression model on unseen validation samples
### 2. Calculate accuracy
### 3. Calculate precision
### 4. Calculate recall
### 5. Calculate F1 score
### 6. Generate a confusion matrix
### 7. Experiment with classification thresholds of 0.3, 0.5, and 0.7
### 8. Observe how precision and recall change
### 9. Plot the learned decision boundary
### 10. Compare the training and validation performance


In [3]:
def predict_proba(X_in, W_in, b_in):
  with torch.no_grad():
    return torch.sigmoid(X_in @ W_in + b_in)

def predict_labels(probs, threshold=0.5):
  return (probs >= threshold).float()

def confusion_counts(y_true, y_pred):
  y_true = y_true.view(-1)
  y_pred = y_pred.view(-1)
  tp = ((y_true == 1) & (y_pred == 1)).sum().item()
  tn = ((y_true == 0) & (y_pred == 0)).sum().item()
  fp = ((y_true == 0) & (y_pred == 1)).sum().item()
  fn = ((y_true == 1) & (y_pred == 0)).sum().item()
  return {'tp': tp, 'tn': tn, 'fp': fp, 'fn': fn}

def metrics_from_counts(c):
  tp, tn, fp, fn = c['tp'], c['tn'], c['fp'], c['fn']
  total = tp + tn + fp + fn
  acc = (tp + tn) / total if total else 0.0
  prec = tp / (tp + fp) if (tp + fp) else 0.0
  rec = tp / (tp + fn) if (tp + fn) else 0.0
  f1 = 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0
  return {'accuracy': acc, 'precision': prec, 'recall': rec, 'f1': f1}

probs_val = predict_proba(X_val, W, b)
probs_train = predict_proba(X_train, W, b)

print('Threshold experiments (validation):')
for t in [0.3, 0.5, 0.7]:
  preds = predict_labels(probs_val, threshold=t)
  c = confusion_counts(y_val, preds)
  m = metrics_from_counts(c)
  print('Threshold', t)
  print('  Confusion matrix counts:', c)
  print('  Accuracy:', m['accuracy'], 'Precision:', m['precision'], 'Recall:', m['recall'], 'F1:', m['f1'])

# Train vs validation at 0.5
for split_name, probs_s, y_s in [('train', probs_train, y_train), ('val', probs_val, y_val)]:
  preds = predict_labels(probs_s, 0.5)
  c = confusion_counts(y_s, preds)
  m = metrics_from_counts(c)
  print('\n{} performance @0.5:'.format(split_name), m, 'counts', c)

# Decision boundary plot
w1, w2 = W.detach().numpy().flatten()
bb = b.detach().item()
x1_min, x1_max = X[:,0].min().item()-0.5, X[:,0].max().item()+0.5
x2_min, x2_max = X[:,1].min().item()-0.5, X[:,1].max().item()+0.5
xx1, xx2 = np.meshgrid(np.linspace(x1_min, x1_max, 200), np.linspace(x2_min, x2_max, 200))
grid = torch.tensor(np.c_[xx1.ravel(), xx2.ravel()], dtype=torch.float32)
with torch.no_grad():
  zz = torch.sigmoid(grid @ W + b).reshape(xx1.shape).numpy()
plt.figure(figsize=(5,4))
plt.contourf(xx1, xx2, zz, levels=20, alpha=0.3, cmap='RdBu')
plt.contour(xx1, xx2, zz, levels=[0.3, 0.5, 0.7], colors=['orange','black','green'], linestyles=['--','-','--'])
plt.scatter(X_train[y_train.squeeze()==0,0].numpy(), X_train[y_train.squeeze()==0,1].numpy(), s=10, alpha=0.5, label='train 0')
plt.scatter(X_train[y_train.squeeze()==1,0].numpy(), X_train[y_train.squeeze()==1,1].numpy(), s=10, alpha=0.5, label='train 1')
plt.scatter(X_val[y_val.squeeze()==0,0].numpy(), X_val[y_val.squeeze()==0,1].numpy(), s=20, marker='x', label='val 0')
plt.scatter(X_val[y_val.squeeze()==1,0].numpy(), X_val[y_val.squeeze()==1,1].numpy(), s=20, marker='x', label='val 1')
plt.title('Decision boundary (thresholds 0.3/0.5/0.7)')
plt.xlabel('x1')
plt.ylabel('x2')
plt.legend(loc='best', fontsize=8)
plt.tight_layout()
plt.savefig('../reports/section14_decision_boundary.png', dpi=120)
plt.show()
print('Saved decision boundary plot to reports/section14_decision_boundary.png')


Threshold experiments (validation):
Threshold 0.3
  Confusion matrix counts: {'tp': 28, 'tn': 32, 'fp': 0, 'fn': 0}
  Accuracy: 1.0 Precision: 1.0 Recall: 1.0 F1: 1.0
Threshold 0.5
  Confusion matrix counts: {'tp': 28, 'tn': 32, 'fp': 0, 'fn': 0}
  Accuracy: 1.0 Precision: 1.0 Recall: 1.0 F1: 1.0
Threshold 0.7
  Confusion matrix counts: {'tp': 28, 'tn': 32, 'fp': 0, 'fn': 0}
  Accuracy: 1.0 Precision: 1.0 Recall: 1.0 F1: 1.0

train performance @0.5: {'accuracy': 1.0, 'precision': 1.0, 'recall': 1.0, 'f1': 1.0} counts {'tp': 122, 'tn': 118, 'fp': 0, 'fn': 0}

val performance @0.5: {'accuracy': 1.0, 'precision': 1.0, 'recall': 1.0, 'f1': 1.0} counts {'tp': 28, 'tn': 32, 'fp': 0, 'fn': 0}


Saved decision boundary plot to reports/section14_decision_boundary.png


/tmp/ipykernel_69101/3195626484.py:67: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
